# Dynamic Wakes 2

## Horizontally homogeneous wake propagation

For horizontally homogeneous timeseries input data (i.e., dependency on time and optionally also on `z`, but _not_ on `x, y` coordinates), `foxes` offers a simplified way to compute dynamic wake propagation. This in principle works by following a flow trace backwards in time from each point of interest, and identifying it with a wake trajectory if it approaches the vicinity of a rotor. For horizontally homogeneous inflow the steps of these traces are independent of the evaluation point.

Similarly to the prevously discussed `DynamicWakes` approach, this concept only works if

- either all states fall into a single chunk,
- or the `Iterative` algorithm is used for the calculation.

These are the inlcudes for this example:

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

import foxes
import foxes.variables as FV
import foxes.constants as FC

from pathlib import Path
from tempfile import TemporaryDirectory
from IPython.display import Image

_animation_dir = TemporaryDirectory(prefix="foxes-animation-")

This is the engine that we are going to use throughout all calculations in this notebook:

In [ ]:
engine = foxes.Engine.new(
    "process", chunk_size_states=20, chunk_size_points=500, n_procs=8
)

We create a case with a regular 3 x 3 wind farm layout:

In [ ]:
states = foxes.input.states.Timeseries(
    data_source="timeseries_100.csv.gz",
    output_vars=[FV.WS, FV.WD, FV.TI, FV.RHO],
    var2col={FV.WS: "ws", FV.WD: "wd", FV.TI: "ti"},
    fixed_vars={FV.RHO: 1.225, FV.TI: 0.07},
)

farm = foxes.WindFarm()
foxes.input.farm_layout.add_grid(
    farm,
    xy_base=np.array([0.0, 0.0]),
    step_vectors=np.array([[1000.0, 0], [0, 800.0]]),
    steps=(3, 3),
    turbine_models=["DTU10MW"],
    verbosity=0,
)

mbook = foxes.ModelBook()
mbook.outputs["create_turbine_gif"] = foxes.models.outputs.FlowTurbineAnimation(
    base_name="timelines.gif",
    out_dir=_animation_dir.name,
    fps=5,
    turbines=[1, 4, 7],
    figsize=(5.2, 7),
    grid_pars=dict(resolution=30, xmax=5000, ymax=5000),
    plot_pars=dict(
        vmin=0,
        quiver_pars=dict(scale=0.013),
        quiver_n=35,
        title={i: f"t = {i / 6:3.2f} min" for i in range(states.size())}.get,
        rotor_color="red",
    ),
    layout_pars=dict(title="", annotate=1, anno_delx=-120, anno_dely=-60, alpha=0),
)

algo = foxes.algorithms.Iterative(
    farm,
    states,
    wake_models=["Bastankhah2014_linear_lim_k004"],
    mbook=mbook,
    wake_frame="timelines",
    max_wake_length_km=8.0,
    verbosity=1,
)

Notice the wake frame choice `timelines`, which is a pre-defined instance of the class `Timelines` from the model book. Also, we choose a maximal wake length to 8 km for this example.

First calculate keyed farm results, then create the animation with `calc_outputs`. We retain `farm_results_all` so the subsequent flow calculation has the required wake inputs, including thrust coefficients:

In [ ]:
with engine:
    results = algo.calc_farm(outputs="farm_results_all", ret_dict=True)
    algo.calc_outputs(
        outputs="create_turbine_gif", farm_results=results[FC.FARM_RESULTS]
    )
farm_results = results[FC.FARM_RESULTS]

Notice the iterations and the convergence behaviour, resulting in less iterations than the previous `DynamicWakes` example. Now the farm results are ready:

In [ ]:
farm_df = farm_results.to_dataframe()
print("\nFarm results data:\n")
print(farm_df[[FV.AMB_REWS, FV.REWS, FV.P]])

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(farm_results[FC.STATE], farm_results[FV.REWS][:, 1], label="Turbine 1")
ax.plot(farm_results[FC.STATE], farm_results[FV.REWS][:, 4], label="Turbine 4")
ax.plot(farm_results[FC.STATE], farm_results[FV.REWS][:, 7], label="Turbine 7")
ax.legend()
ax.set_xlabel("Time")
ax.set_ylabel("REWS [m/s]")
plt.show()

This timeseries has a time step of 1 minute. Let's visualize the wake dynamics in an animation: The `create_turbine_gif` model-book output writes the animation as part of the final farm evaluation.

In [ ]:
# The writer appends the animated variable to the configured base filename.
animation = mbook.outputs["create_turbine_gif"]
path = Path(animation.base_name)
Image(filename=str(animation.get_fpath(f"{path.stem}_WS{path.suffix}")))

In [ ]:
plt.close("all")

In [ ]:
# GIF data is embedded in the preceding display.

In [ ]:
del farm_results
plt.close("all")

For the fun of it, let's re-run this case assuming the time step was 10 s instead of 1 min. We can do so by using the wake frame `Timelines(dt_min=1/6)`, which is called `timelines_10s` in the model book. We also select a maximal wake length of 8 km in this example:

In [ ]:
mbook = foxes.ModelBook()
mbook.outputs["create_turbine_gif"] = foxes.models.outputs.FlowTurbineAnimation(
    base_name="timelines_10s.gif",
    out_dir=_animation_dir.name,
    fps=5,
    turbines=[1, 4, 7],
    figsize=(5.2, 7),
    grid_pars=dict(resolution=30, xmax=5000, ymax=5000),
    plot_pars=dict(
        vmin=0,
        quiver_pars=dict(scale=0.013),
        quiver_n=35,
        title={i: f"t = {i / 6:3.2f} min" for i in range(states.size())}.get,
        rotor_color="red",
    ),
    layout_pars=dict(title="", annotate=1, anno_delx=-120, anno_dely=-60, alpha=0),
)

algo = foxes.algorithms.Iterative(
    farm,
    states,
    wake_models=["Bastankhah2014_linear_lim_k004"],
    mbook=mbook,
    wake_frame="timelines_10s",
    max_wake_length_km=8.0,
    verbosity=1,
)


with engine:
    results = algo.calc_farm(outputs="farm_results_all", ret_dict=True)
    algo.calc_outputs(
        outputs="create_turbine_gif", farm_results=results[FC.FARM_RESULTS]
    )
farm_results = results[FC.FARM_RESULTS]

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(farm_results[FC.STATE], farm_results[FV.REWS][:, 1], label="Turbine 1")
ax.plot(farm_results[FC.STATE], farm_results[FV.REWS][:, 4], label="Turbine 4")
ax.plot(farm_results[FC.STATE], farm_results[FV.REWS][:, 7], label="Turbine 7")
ax.legend()
ax.set_xlabel("Time")
ax.set_ylabel("REWS [m/s]")
plt.show()

In [ ]:
# The writer appends the animated variable to the configured base filename.
animation = mbook.outputs["create_turbine_gif"]
path = Path(animation.base_name)
Image(filename=str(animation.get_fpath(f"{path.stem}_WS{path.suffix}")))

In [ ]:
_animation_dir.cleanup()
plt.close("all")